In [1]:
"""Step 1: collect public Google Play reviews for Kenyan finance apps.
 
Run this on your own machine (needs internet access to Google Play).
Find each package id in the app's Play Store URL: ...details?id=<PACKAGE_ID>
Verify every id yourself before running.
"""
import hashlib, os, time
import pandas as pd
from google_play_scraper import reviews, Sort
 

In [4]:

LANGS = ["en", "sw"]          # Play Store language filter
MAX_PER_APP_LANG = 3000
SALT = os.environ.get("REVIEW_SALT", "change-me")  # keep secret, never commit
 

In [5]:
APPS = {
    # name: package id  -> FILL IN / VERIFY from the Play Store URL
    "mpesa_or_mysafaricom": "com.safaricom.mpesa.orgapp",
    "tala": "com.inventureaccess.safarirahisi",
    "branch": "com.branch_international.branch.branch_demo_android",
    "kcb": "com.kcb.mobilebanking.android.mbp",
    "equity": "ke.co.equitygroup.equitymobile",
    "coop": "com.mcoopcash.retail",
    "absa_ke": "com.barclays.absa.banking",
}

## What this data supports: 
- **Complaint classification:** yes. Single reviews with categories like failed transactions, fees, app problems and customer service are what TF-IDF plus logistic regression needs, provided you reach a few hundred labelled examples per major class.
- **Sentiment:** yes, for English and Swahili, with RideKE and AfriSenti adding Swahili and code-switched coverage.
- **Explainability (SHAP/LIME):** yes, because it works on any of these classifiers.

In [ ]:
LANGS = ["en", "sw"]          # Play Store language filter
# Newest-first returns mostly short 5-star praise, so collect per star rating.
# Quotas are per app and per language. Low stars are where complaints live.
QUOTA = {1: 1500, 2: 800, 3: 600, 4: 200, 5: 200}
SALT = os.environ.get("REVIEW_SALT", "change-me")  # keep secret, never commit
 
rows = []
for app, pkg in APPS.items():
    if pkg == "PACKAGE_ID_HERE":
        continue
    for lang in LANGS:
        for stars, quota in QUOTA.items():
            token, got = None, 0
            while got < quota:
                batch, token = reviews(pkg, lang=lang, country="ke",
                                       sort=Sort.NEWEST, count=200,
                                       filter_score_with=stars,
                                       continuation_token=token)
                for r in batch:
                    rows.append({
                        # hashed id instead of reviewId/userName (no usernames stored)
                        "uid": hashlib.sha256((SALT + r["reviewId"]).encode()).hexdigest()[:16],
                        "app": app, "source": "google_play", "lang_filter": lang,
                        "rating": r["score"], "date": r["at"].date().isoformat(),
                        "text": r["content"],
                        "dev_replied": r["replyContent"] is not None,
                    })
                got += len(batch)
                if not token or not batch:
                    break
                time.sleep(1.5)  # be polite
 

In [ ]:
df = pd.DataFrame(rows).drop_duplicates("uid")
df.to_csv("raw_playstore_reviews.csv", index=False)   # RAW: do not share
print(len(df), "reviews saved")

21858 reviews saved


In [11]:
df.head()

,uid,app,source,lang_filter,rating,date,text,dev_replied
0,72efdb74d0a8a2d4,mpesa_or_mysafaricom,google_play,en,1,2026-10-16,The app requires you to DISABLE developer opti...,False
1,5570bec1d1a60817,mpesa_or_mysafaricom,google_play,en,1,2026-10-16,horrible,False
2,cb36c1c15e283bc5,mpesa_or_mysafaricom,google_play,en,1,2026-10-11,"My app not able to open, very disappointed.",False
3,f26eeb26d2c72010,mpesa_or_mysafaricom,google_play,en,1,2026-10-10,amazing,False
4,32e5d40a0cde7088,mpesa_or_mysafaricom,google_play,en,1,2026-10-08,It's better not using it at all. I'm tired of ...,False


In [14]:
df["rating"].value_counts()

rating
1    10932
2     4499
3     3450
5     1632
4     1345
Name: count, dtype: int64

- Swahili is little

What to do about Swahili

- Put every review with Swahili or Sheng markers into your labelling set, not just a random sample. That's probably a few hundred rows, enough for a small per-language evaluation.
- Use AfriSenti and RideKE for Swahili and code-switched sentiment.
- If after labelling Swahili, Sheng and mixed text is under about 10% of the set, state that the complaint-category and escalation results are English-dominant and the Swahili results are preliminary.